In [1]:
""" Create the observational data set after controlling for pretreatment effects
    "extract_obs_productivity.ipynb" only applies it on biomass

    Because of the considerable pre-treatment effect reported in Hanson et al. 2025,
    we also remove the pre-treatment effect from AGNPP of the trees

    Hanson, P. J., Griffiths, N. A., Salmon, V. G., Birkebak, J. M., Warren, J. M., Phillips, J. R., et al. (2025). Peatland Plant Community Changes in Annual Production and Composition Through 8 Years of Warming Manipulations Under Ambient and Elevated CO2 Atmospheres. Journal of Geophysical Research: Biogeosciences, 130(2), e2024JG008511. https://doi.org/10.1029/2024JG008511
"""
import pandas as pd
import os
import numpy as np
import statsmodels.formula.api as smf

In [ ]:
obs_paul = pd.read_excel(os.path.join(os.environ['HOME'],
    'Git', 'phenology_elm', "SPRUCE C Budget Summary 28Apr2022EXP.xlsx"),
    sheet_name="DataForPythonRead", skiprows=1,  engine="openpyxl")
obs_paul = obs_paul.loc[obs_paul["Year"] != 2020, :]
obs_paul = obs_paul.set_index(['Plot', 'Year']).sort_index()

obs_verity = pd.read_csv(os.path.join(os.environ['HOME'],
    'Git', 'phenology_elm', 'SalmonSPRUCE_2016to2021_AbovegroundPFT_CNPbudget_20240208.csv'))
# match by plot and year to temperature
obs_verity['Plot'] = [f'P{p:02d}' for p in obs_verity['Plot']]
obs_verity = obs_verity.set_index(['Plot', 'Year', 'PFT']).unstack()
obs_verity = obs_verity.loc[:, (slice(None), 
                                ['Sphagnum', 'evergreen conifer', 'deciduous conifer',  
                                    'shrub'])]
obs_verity2 = obs_verity.loc[:, ['ABGbiomass_gCperm2', 'ABGnpp_gCperm2peryear',
                                 'Pretrt_ABGbiomass_gCperm2', 'Pretrt_ABGnpp_gCperm2peryear']]
obs_data = pd.concat([obs_paul, obs_verity2], axis = 1, join = 'outer')
obs_data = obs_data.reset_index()

# Merge 0 and Amb
obs_data['eCO2'] = np.where(obs_data['CO2'].isna(), np.nan, obs_data['CO2'] == 500)

In [3]:
obs_data2 = pd.DataFrame(np.nan, 
    index = pd.MultiIndex.from_frame(obs_data[['Plot','Year']]),
    columns = ['eCO2', 'Tair', 
               'AGBiomass_Spruce', 'AGBiomass_Tamarack', 'AGBiomass_Shrub', # gC m-2
               'AGNPPtoBiomass_Spruce', 'AGNPPtoBiomass_Tamarack', 'AGNPPtoBiomass_Shrub', # NPP to Biomass ratio, yr-1
               'AGNPP_Spruce', 'AGNPP_Tamarack', 'AGNPP_Shrub', 'NPP_moss' # gC m-2 yr-1
               'BGNPP_TreeShrub', # fine root NPP, gC m-2 yr-1
               'BGtoAG_TreeShrub', # fine root NPP to AGNPP ratio
               'AGNPP', # tree + shrub + moss
               'HR', 'NEE']
)

In [4]:
obs_data2.loc[:, 'eCO2'] = obs_data['eCO2'].values
obs_data2.loc[:, 'Tair'] = obs_data.loc[:, 'Mean Annual Temp. at 2 m'].values
obs_data2.loc[:, 'BGNPP_TreeShrub'] = obs_data['BNPP Tree & Shrub'].values
obs_data2.loc[:, 'HR'] = obs_data.loc[:, 'RHCO2'].values
obs_data2.loc[:, 'NEE'] = obs_data.loc[:, 'NCE'].values
obs_data2.loc[:, 'NPP_moss'] = obs_data.loc[:, 'NPP Sphag.'].values

In [5]:
# Use linear regression to remove pre-treatment effect on the
# aboveground biomass & aboveground NPP of trees
# 
# Post treatment level ~ I(eCO2) + beta1 * Tair
#    + beta2 * Pre treatment level + beta3 * I(eCO2) x Tair
#    + beta4 * I(eCO2) x Pre treatment level + beta 5 * Tair x Pre treatment level
# 
# Subtract out all three terms that include pretreatment effect if significant

def create_X(obs_data, varpre, varname, pft2):
    temp_df = {
        'eCO2': obs_data['eCO2'],
        'Tair': obs_data['Mean Annual Temp. at 2 m'].values,
        'Pretrt': obs_data[(varpre,pft2)].values,
        'Postrt': obs_data[(varname,pft2)].values,
    }
    temp_df = pd.DataFrame(temp_df)
    temp_df = temp_df.dropna(how = 'any', axis = 0)

    return temp_df


for varname, varpre, varfinal in zip(['ABGbiomass_gCperm2', 'ABGnpp_gCperm2peryear'],
                                     ['Pretrt_ABGbiomass_gCperm2', 'Pretrt_ABGnpp_gCperm2peryear'],
                                     ['AGBiomass', 'AGNPP']):
    for pft,pft2 in zip(['Spruce', 'Tamarack', 'Shrub'],
                        ['evergreen conifer','deciduous conifer','shrub']):

        # Form the dataset
        df = create_X(obs_data, varpre, varname, pft2)

        # Full model
        full_formula = ("Postrt ~ eCO2 + Tair + Pretrt + eCO2:Tair + eCO2:Pretrt + Tair:Pretrt")
        full_mod = smf.ols(full_formula, data=df).fit()

        # Drop insignificant terms and refit
        keepers = [
            term for term, p in full_mod.pvalues.items()
            if (term == "Intercept") or (p <= 0.05)
        ]

        # No pretreatment effect, then subtract nothing
        if not ('Pretrt' in keepers or 'eCO2:Pretrt' in keepers or 'Tair:Pretrt' in keepers):
            obs_data2.loc[:, f'{varfinal}_{pft}'] = obs_data[(varname, pft2)].values

            # It is not possible to perform a sensible estimation of this uncertainty!!!!!!!
            # Just go with proportional to Paul's numbers

            ### Calculate uncertainty in data as simple standard deviation in the
            ### pretreatment values
            ###std0 = obs_data.loc[obs_data['Year'] == 2016, :][(varpre, pft2)].std()
            ###print(varname, pft, 'aCO2 std=', std0, 'eCO2 std=', std0)

            continue

        # Has pretreatment effect, fit reduced model
        reduced_formula = "Postrt ~ " + " + ".join(k for k in keepers if k != "Intercept")
        results = smf.ols(reduced_formula, data=df).fit()

        # Subtract the pretreatment effect if significant
        # (add back the mean of the pretreatment level)
        baseline = 0
        dpretrt = obs_data[(varpre, pft2)].values - obs_data[(varpre, pft2)].mean()
        if 'Pretrt' in results.pvalues.index and results.pvalues['Pretrt'] <= 0.05:
            baseline += results.params['Pretrt'] *dpretrt
        if 'eCO2:Pretrt' in results.pvalues.index and results.pvalues['eCO2:Pretrt'] <= 0.05:
            baseline += results.params['eCO2:Pretrt'] * dpretrt * obs_data['eCO2'].values
        if 'Tair:Pretrt' in results.pvalues.index and results.pvalues['Tair:Pretrt'] <= 0.05:
            baseline += results.params['Tair:Pretrt'] * dpretrt * \
                        obs_data['Mean Annual Temp. at 2 m'].values

        obs_data2.loc[:, f'{varfinal}_{pft}'] = obs_data[(varname, pft2)].values - baseline

        # It is not possible to perform a sensible estimation of this uncertainty!!!!!!!
        # Just go with proportional to Paul's numbers

        ## Print the uncertainty in the subtracted value
        ## - ambient CO2
        #var0 = np.sqrt(results.bse['Intercept']**2 + \
        #       results.bse['Pretrt']**2 * obs_data[(varpre, pft2)].mean() + \
        #       results.params['Pretrt']**2 * obs_data[(varpre, pft2)].var())
        #print(varname, pft, gamma_list, var0)

In [6]:
for pft,pft2 in zip(['Spruce', 'Tamarack', 'Shrub'], 
                    ['evergreen conifer','deciduous conifer','shrub']):
    obs_data2.loc[:, f'AGNPPtoBiomass_{pft}'] = \
        obs_data[('ABGnpp_gCperm2peryear', pft2)].values / \
        obs_data[('ABGbiomass_gCperm2', pft2)].values

In [7]:
obs_data2.loc[:, 'BGtoAG_TreeShrub'] = \
    obs_data2.loc[:, 'BGNPP_TreeShrub'].values / \
    (obs_data2.loc[:, 'AGNPP_Spruce'] + obs_data2.loc[:, 'AGNPP_Tamarack'] + \
     obs_data2.loc[:, 'AGNPP_Shrub']).values

In [8]:
obs_data2.loc[:, 'NPP'] = \
    (obs_data2.loc[:, 'AGNPP_Spruce'] + obs_data2.loc[:, 'AGNPP_Tamarack'] + \
     obs_data2.loc[:, 'AGNPP_Shrub'] + obs_data2.loc[:, 'BGNPP_TreeShrub'] + \
     obs_data2.loc[:, 'NPP_moss']).values

In [9]:
path_out = os.path.join(os.environ['PROJDIR'], 'ELM_Phenology', 'output', 'extract')
obs_data2.to_csv(os.path.join(path_out, 'extract_obs_productivity.csv'))

obs_data2

eCO2       Tair  AGBiomass_Spruce  AGBiomass_Tamarack  \
Plot Year                                                          
P04  2016   1.0  12.700000        971.605207          210.150569   
     2017   1.0  11.300000       1039.226207          228.755116   
     2018   1.0  10.800000       1128.137207          245.006382   
     2019   1.0   9.967064       1189.958207          268.892913   
     2021   1.0  12.100000       1260.286207          254.976089   
...         ...        ...               ...                 ...   
P13  2020   NaN        NaN       1091.384107                 NaN   
P16  2020   NaN        NaN       1051.702447                 NaN   
P17  2020   NaN        NaN        855.141240                 NaN   
P19  2020   NaN        NaN       1046.081468                 NaN   
P20  2020   NaN        NaN       1354.180280                 NaN   

           AGBiomass_Shrub  AGNPPtoBiomass_Spruce  AGNPPtoBiomass_Tamarack  \
Plot Year                                                                    
P04  2016          354.660               0.161327                 0.007305   
     2017          418.203               0.099601                 0.109131   
     2018          273.278               0.150335                 0.145624   
     2019          311.169               0.121375                 0.165687   
     2021          423.661               0.128676                 0.108055   
...                    ...                    ...                      ...   
P13  2020              NaN               0.104109                 0.129768   
P16  2020              NaN               0.070416                 0.139325   
P17  2020              NaN               0.076110                 0.194170   
P19  2020              NaN               0.062666                 0.103896   
P20  2020              NaN               0.099630                 0.152179   

           AGNPPtoBiomass_Shrub  AGNPP_Spruce  AGNPP_Tamarack  AGNPP_Shrub  \
Plot Year                                                                    
P04  2016              0.499774    110.260242       -1.795054      177.250   
     2017              0.390753     87.727242       18.775398      163.414   
     2018              0.343566    128.580242       28.423274       93.889   
     2019              0.406605    117.819242       36.375923      126.523   
     2021              0.354991    131.924242       21.821997      150.396   
...                         ...           ...             ...          ...   
P13  2020                   NaN           NaN             NaN          NaN   
P16  2020                   NaN           NaN             NaN          NaN   
P17  2020                   NaN           NaN             NaN          NaN   
P19  2020                   NaN           NaN             NaN          NaN   
P20  2020                   NaN           NaN             NaN          NaN   

           NPP_mossBGNPP_TreeShrub  BGtoAG_TreeShrub  AGNPP         HR  \
Plot Year                                                                
P04  2016                      NaN          0.367149    NaN -537.30000   
     2017                      NaN          0.403087    NaN -453.30000   
     2018                      NaN          0.425880    NaN -456.80000   
     2019                      NaN          0.384104    NaN -381.07052   
     2021                      NaN          0.353720    NaN -453.75570   
...                            ...               ...    ...        ...   
P13  2020                      NaN               NaN    NaN        NaN   
P16  2020                      NaN               NaN    NaN        NaN   
P17  2020                      NaN               NaN    NaN        NaN   
P19  2020                      NaN               NaN    NaN        NaN   
P20  2020                      NaN               NaN    NaN        NaN   

                  NEE  BGNPP_TreeShrub  NPP_moss         NPP  
Plot Year                                                     
P04  20